# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to explore and analyze the FAIR² dataset on mass spectrometry analysis of extracellular vesicles from human mast cells using the [`mlcroissant`](https://mlcommons.org/croissant/) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Install the mlcroissant library if necessary
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset schema and metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
List the available record sets, their IDs, and the fields present in each. All references use `@id` for clarity and reproducibility.

Below, we display the main record sets, their IDs, and all field and column `@id`s in each set.

In [ ]:
# Extract the metadata for record sets and fields
record_sets = list(dataset.record_sets)

print("Available record sets:")
for rs in record_sets:
    print(f"\nRecord Set name:   {rs.name}")
    print(f"Record Set @id:    {rs.id}")
    print("Fields (as @id:s and labels):")
    for field in rs.fields:
        col_str = f"  (column @id: {field.column.id})" if field.column else ""
        print(f"   - Field @id: {field.id} | label: {field.label}{col_str}")

## 3. Data Extraction
Load records from selected record sets into pandas DataFrames for exploration. Reference entities using their `@id`s for traceability.

In [ ]:
# For this dataset, the main tabular record set contains the protein analysis table.
# We'll extract all record set @ids and load their data.
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = dict()

# Load all records in all record sets as DataFrames
# (Many datasets will have only one record set, but we generalize for completeness)
for record_set_id in record_set_ids:
    print(f"Loading record set: {record_set_id}")
    records_iter = dataset.records(record_set=record_set_id)
    records = list(records_iter)
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"{len(df)} rows loaded. Columns: {df.columns.tolist()}")
    # Show the top of the DataFrame
    print(df.head(2))

## 4. Exploratory Data Analysis (EDA)
Let's process the primary protein abundance record set. We'll:
- Select a numeric field (e.g., 'cr:abundance' or similar, by its `@id` as referenced in the prior section),
- Filter records by a threshold,
- Normalize the numeric values,
- Optionally, group by a categorical field if present.

> **Note**: Replace the field IDs with exact values from Section 2 above. You can also run the code blocks so the true IDs and column names are printed for clarity.

In [ ]:
# Choose the main record set to demonstrate EDA (choose the first one for this example)
main_record_set_id = record_set_ids[0]
df = dataframes[main_record_set_id]

# Find a suitable numeric field for analysis
# For demonstration, search columns for likely numeric field names (e.g., containing 'count','peptide','abundance','coverage','MW','_norm','intensity')
numeric_candidates = [col for col in df.columns if any(x in col.lower() for x in ['count','peptide','abundance','coverage','mw','intensity','norm'])]
print("Numeric field candidates:", numeric_candidates)

# Let's try using the first numeric candidate, or fallback to a known ID
if numeric_candidates:
    numeric_field = numeric_candidates[0]
else:
    # Replace with a known field @id if no match found
    numeric_field = df.columns[0]
print('Selected numeric_field:', numeric_field)

# Example: Filter for value > threshold
threshold = df[numeric_field].quantile(0.8)  # Use 80th percentile for demonstration
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold:.2f} (top 20%): {len(filtered_df)} rows")
print(filtered_df[[numeric_field]].head())

# Normalize the selected numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a categorical field if present (such as @id or label including 'sample', 'type', etc.)
group_candidates = [col for col in df.columns if any(x in col.lower() for x in ['type', 'sample', 'group', 'condition'])]
if group_candidates:
    group_field = group_candidates[0]
    print(f"\nGrouping results by field: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(grouped_df.head())
else:
    print("No obvious grouping field found.")

## 5. Visualization
Visualize distributions of the chosen numeric field, and compare groups if possible. Adjust the field and grouping variables as needed based on your data.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.show()

# If group_field exists, show boxplot
if 'group_field' in locals():
    plt.figure(figsize=(7,4))
    sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field])
    plt.title(f'{numeric_field} by {group_field}')
    plt.show()

## 6. Conclusion
- Loaded the FAIR² protein abundance dataset defined via Croissant schema.
- Explored available record sets, fields, and columns using their `@id` entries for reliability.
- Performed exploratory data analysis (EDA), including filtering, normalization, and grouping operations.
- Created basic visualizations for the main numeric field.

For further analysis, you can explore relationships between other fields, perform more advanced normalizations, or integrate with bioinformatics pipelines.

_This workflow can be adapted for other Croissant-conformant datasets and extended to machine learning, statistical analysis, or domain-specific modelling._